# SAME latent space

SAME is the autoencoder Stable Audio 3 generates into, so it is the space a prior would live in. Two questions:

1. **Round-trip** — what does `decode(encode(x))` cost us?
2. **Transport** — is MP3 damage a constant offset in that space? If so, `decode(encode(x) + v)` is the cheapest restoration imaginable, and the floor everything else must beat.

Run `scripts/run_same_experiments.py` first. Audio here is level-matched to −14 LUFS with one common headroom gain, so nothing clips and no comparison is decided by loudness.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

sys.path.insert(0, "../src")
from grooveback import audio as ga

ART = Path("../artifacts/same")
RESULTS = json.loads((ART / "results.json").read_text())


def play(directory, order=None):
    """Play a listening set, filename first so it is clear what produced it."""
    files = sorted(Path(directory).glob("*.wav"))
    rank = lambda p: order.index(p.stem) if order and p.stem in order else 99  # noqa: E731
    for path in sorted(files, key=rank):
        print(path.stem)
        display(Audio(str(path)))


def spectra(paths_and_labels, title, floor=-110):
    fig, ax = plt.subplots(figsize=(11, 4))
    for path, label in paths_and_labels:
        x, sr = ga.load(path)
        mono = x.mean(0)
        spec = np.abs(np.fft.rfft(mono * np.hanning(len(mono))))
        freqs = np.fft.rfftfreq(len(mono), 1 / sr)
        # smooth into 1/12-octave bins so the plot is readable
        edges = np.geomspace(20, sr / 2, 120)
        idx = np.digitize(freqs, edges)
        binned = np.array([spec[idx == i].mean() if (idx == i).any() else np.nan
                           for i in range(1, len(edges))])
        ax.semilogx(edges[:-1], 20 * np.log10(binned / len(mono) + 1e-12), label=label)
    ax.set(xlim=(20, 22050), ylim=(floor, None), xlabel="Hz", ylabel="dB", title=title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.show()

## 1. Round-trip

Excerpts are 12 s taken at 1:00 and 3:00 of each source — one fixed rule, intro and main body. `codec` is a 6 s clean asset whose content stops around 5 kHz, kept as the severe case.

`an2` is a real 128 kbps YouTube rip with no clean master.

Residual is `input − output`, saved before any level matching, so it is the honest difference.

In [ ]:
bands = list(next(iter(RESULTS["roundtrip"].values()))["band_delta_db"])
print(f"{'excerpt':22s}{'resid':>7s}" + "".join(f"{b.split('-')[0]:>8s}" for b in bands))
print(f"{'':22s}{'dB':>7s}" + "".join(f"{'Hz':>8s}" for _ in bands))
for key in sorted(RESULTS["roundtrip"]):
    row = RESULTS["roundtrip"][key]
    print(f"{key:22s}{row['residual_below_signal_db']:7.1f}"
          + "".join(f"{row['band_delta_db'][b]:8.2f}" for b in bands))

Below 16 kHz the round-trip is energy-transparent on clean material. Above it, two things happen — and the second is the one that matters.

In [ ]:
spectra([(ART / "roundtrip/aerofunk_60s/1_input.wav", "input (clean master)"),
         (ART / "roundtrip/aerofunk_60s/2_output_decode_encode_same_s.wav",
          "decode(encode(x)) same-s"),
         (ART / "roundtrip/aerofunk_60s/2_output_decode_encode_same_l.wav",
          "decode(encode(x)) same-l")],
        "Clean master: round-trip is close, and adds content above 20 kHz")

spectra([(ART / "roundtrip/an2_60s/1_input.wav", "input (real 128k rip)"),
         (ART / "roundtrip/an2_60s/2_output_decode_encode_same_s.wav", "decode(encode(x)) same-s"),
         (ART / "roundtrip/an2_60s/2_output_decode_encode_same_l.wav", "decode(encode(x)) same-l")],
        "Real rip: the decoder invents a top end that was never there")

**The decoder hallucinates.** The rip is dead above 16 kHz; the round-trip puts plausible content back. Nothing asked it to.

So `decode(encode(degraded))` is not a no-op, and any latent-space method has to be judged against *it*, never against the input file — otherwise the autoencoder's own behaviour gets credited to the method.

Listen to the round-trip, then to the residual, which is what the round-trip threw away or added.

In [ ]:
play(ART / "roundtrip/an2_60s")

## 2. Transport vector

Take a clean master, run it through a 128 kbps LAME encode and back, and encode both to latents. The mean of `z_clean − z_degraded` over many windows is a single 256-dim vector `v`.

The whole method is `decode(encode(x) + v)`. One addition, no model, no iteration.

`v` is fitted on Aerofunk windows only, with the excerpts below held out.

In [ ]:
for bitrate, block in RESULTS["transport"].items():
    s = block["shift"]
    print(f"{bitrate}: |v|={s['norm']:.2f}  variance explained={s['variance_explained']:.0%}  "
          f"direction agreement={s['cosine_mean']:.2f} (min {s['cosine_min']:.2f})  "
          f"from {s['pool_windows']} windows")

Windows agree strongly on the **direction** of the damage but a constant only explains about a third of its **magnitude** — the rest is content-dependent. The 192k vector points the same way at half the length, which is what you would want if this were measuring something real about bitrate.

The 16–20 kHz band, relative to the clean master:

In [ ]:
BAND = "16000-20000"
for bitrate, block in RESULTS["transport"].items():
    print(f"\n{bitrate}   (dB in {BAND} Hz)")
    cols = f"  {'excerpt':16s}{'input':>9s}{'no shift':>10s}{'+ shift':>10s}{'ceiling':>10s}"
    print(cols + "   scored against")
    for name, rows in block["excerpts"].items():
        ceiling = rows.get("ref_decode_encode_clean", {}).get(BAND)
        print(f"  {name:16s}{rows['1_input_mp3'][BAND]:9.2f}"
              f"{rows['ref_decode_encode_input'][BAND]:10.2f}"
              f"{rows['2_output_decode_encode_plus_shift'][BAND]:10.2f}"
              f"{'         —' if ceiling is None else f'{ceiling:10.2f}'}"
              f"   {rows['scored_against']}")

`ceiling` is `decode(encode(clean))` — the best any latent-space method can do, since it never leaves the space.

**AN-2 has no clean master**, so it is scored against its own unshifted round-trip: the question there is only whether a vector fitted on synthetic Aerofunk twins does something sane on a real YouTube rip from a different track.

Before and after the transformation:

In [ ]:
spectra([(ART / "transport/aerofunk_60s_128k/1_input_mp3.wav", "1 input (128k mp3)"),
         (ART / "transport/aerofunk_60s_128k/2_output_decode_encode_plus_shift.wav",
          "2 output: decode(encode(x) + v)"),
         (ART / "transport/aerofunk_60s_128k/ref_decode_encode_input.wav",
          "ref: same path, no shift"),
         (ART / "transport/aerofunk_60s_128k/ref_clean_master.wav", "ref: clean master"),
         (ART / "transport/aerofunk_60s_128k/ref_decode_encode_clean.wav", "ref: ceiling")],
        "aerofunk_60s @ 128k — held out from the fit")

spectra([(ART / "transport/an2_60s_128k/1_input_mp3.wav", "1 input (real 128k rip)"),
         (ART / "transport/an2_60s_128k/2_output_decode_encode_plus_shift.wav",
          "2 output: decode(encode(x) + v)"),
         (ART / "transport/an2_60s_128k/ref_decode_encode_input.wav", "ref: same path, no shift")],
        "an2_60s @ 128k — different track, real rip, vector fitted elsewhere")

In [ ]:
ORDER = ["1_input_mp3", "2_output_decode_encode_plus_shift",
         "ref_decode_encode_input", "ref_clean_master", "ref_decode_encode_clean"]
play(ART / "transport/aerofunk_60s_128k", ORDER)

In [ ]:
play(ART / "transport/an2_60s_128k", ORDER)

## What to listen for

Energy is not perception, and every number above is energy. Specifically:

- Does `2_output` sound **better** than `1_input`, or just brighter?
- Does it sound better than `ref_decode_encode_input` — i.e. is the shift doing anything the autoencoder was not already doing on its own?
- On AN-2, does a vector fitted on a different track make it sound restored, or just tilted?

Whatever a prior does later has to beat this. One 256-float addition is the floor.